# 16 — Regularized LightGBM threshold analysis

This notebook analyzes decision thresholds for the saved diagnostic regularized LightGBM using validation data only. It does not retrain or calibrate the model and does not load or evaluate test data.

### What this cell does
Maps the saved regularized model, validation inputs, identifiers, parameters, and robustness reports, then fingerprints every protected file used by the analysis.

### Why it matters
Threshold analysis must use the validated model unchanged and must prove that no model, preprocessing object, split, or prior report is overwritten.

### What to understand
Only validation rows and raw model probabilities are used. No test artifact is opened or inspected.

In [1]:
from pathlib import Path
import hashlib
import json

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
assert ROOT.name == "AdoptAI_V1"
PREPROCESSED_DIR = ROOT / "data/modeling/preprocessed"
REPORT_DIR = ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures/threshold_optimization"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    "regularized_model": ROOT / "models/diagnostic/regularized_lightgbm.joblib",
    "regularized_parameters": ROOT / "models/diagnostic/regularized_lightgbm_parameters.json",
    "X_validation": PREPROCESSED_DIR / "X_validation_tree.csv",
    "y_validation": PREPROCESSED_DIR / "y_validation.csv",
    "validation_identifiers": PREPROCESSED_DIR / "validation_identifiers.csv",
    "tree_preprocessor": ROOT / "models/preprocessing/tree_preprocessor.joblib",
    "global_comparison": REPORT_DIR / "regularized_lgbm_global_comparison.csv",
    "run_comparison": REPORT_DIR / "regularized_lgbm_by_run.csv",
    "machine_comparison": REPORT_DIR / "regularized_lgbm_by_machine.csv",
    "stability_summary": REPORT_DIR / "regularized_lgbm_stability_summary.csv",
}
assert all(path.is_file() for path in paths.values())

def sha256_file(path):
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda:handle.read(1024*1024),b""): digest.update(block)
    return digest.hexdigest()
hashes_before={name:sha256_file(path) for name,path in paths.items()}
print(f"Protected artifacts fingerprinted: {len(hashes_before)}")
print("Validation-only boundary established: no test artifact is referenced or loaded.")

Protected artifacts fingerprinted: 10
Validation-only boundary established: no test artifact is referenced or loaded.


### What this cell does
Loads the validation matrix, target, identifiers, saved diagnostic model, complete parameter dictionary, and prior comparison metadata; then generates raw validation probabilities.

### Why it matters
Probability and row validation confirms that threshold results belong to the exact regularized model from notebook 15 and the official 6,640 validation rows.

### What to understand
Probabilities are finite values between zero and one, and the target retains the unusual 65.18% validation prevalence.

In [2]:
X_validation=pd.read_csv(paths["X_validation"])
y_validation=pd.read_csv(paths["y_validation"])["slowdown_in_5min"].astype(int)
validation_identifiers=pd.read_csv(paths["validation_identifiers"])
regularized_model=joblib.load(paths["regularized_model"])
saved_parameters=json.loads(paths["regularized_parameters"].read_text())
global_reference=pd.read_csv(paths["global_comparison"])
prior_run_comparison=pd.read_csv(paths["run_comparison"])
assert isinstance(regularized_model,LGBMClassifier)
for key,value in saved_parameters.items():
    actual=regularized_model.get_params().get(key)
    assert actual==value or (isinstance(value,float) and np.isclose(actual,value)),(key,actual,value)
assert X_validation.shape==(6_640,360) and len(y_validation)==len(validation_identifiers)==6_640
assert set(y_validation.unique())=={0,1} and np.isclose(y_validation.mean(),0.6518072289)
assert not X_validation.isna().any().any() and np.isfinite(X_validation.to_numpy(dtype=float)).all()
validation_probability=regularized_model.predict_proba(X_validation)[:,1]
assert len(validation_probability)==len(y_validation) and np.isfinite(validation_probability).all()
assert ((validation_probability>=0)&(validation_probability<=1)).all()
reference_pr=float(global_reference.loc[global_reference["model"].eq("regularized_lightgbm"),"validation_pr_auc"].iloc[0])
print(f"Validation rows: {len(y_validation):,}; positive rate={y_validation.mean():.2%}")
print(f"Probability range: {validation_probability.min():.6f} to {validation_probability.max():.6f}")
print(f"Validated model reference PR-AUC from notebook 15: {reference_pr:.6f}")
print("No test features, labels, predictions, or test-derived summaries were loaded.")

Validation rows: 6,640; positive rate=65.18%
Probability range: 0.000896 to 0.999959
Validated model reference PR-AUC from notebook 15: 0.989379
No test features, labels, predictions, or test-derived summaries were loaded.


### What this cell does
Evaluates thresholds from 0.01 through 0.99 and records precision, recall, F1, confusion counts, predicted prevalence, error rates, specificity, and balanced accuracy.

### Why it matters
A fine fixed grid exposes the operational tradeoff without optimizing accuracy or changing model probabilities.

### What to understand
Lower thresholds generally reduce false negatives but create more false alerts; higher thresholds do the opposite.

In [3]:
threshold_grid=np.round(np.arange(0.01,1.00,0.01),2)
def threshold_metrics(y_true,probability,threshold):
    prediction=(probability>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,prediction,labels=[0,1]).ravel()
    recall=recall_score(y_true,prediction,zero_division=0); specificity=tn/(tn+fp) if tn+fp else np.nan
    return {"threshold":float(threshold),"precision":precision_score(y_true,prediction,zero_division=0),
            "recall":recall,"f1":f1_score(y_true,prediction,zero_division=0),
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
            "predicted_positive_rate":float(prediction.mean()),
            "false_positive_rate":fp/(fp+tn) if fp+tn else np.nan,
            "false_negative_rate":fn/(fn+tp) if fn+tp else np.nan,
            "specificity":specificity,"balanced_accuracy":float((recall+specificity)/2)}
threshold_table=pd.DataFrame([threshold_metrics(y_validation,validation_probability,t) for t in threshold_grid])
threshold_metrics_path=REPORT_DIR/"threshold_metrics_regularized_lgbm.csv"
threshold_table.to_csv(threshold_metrics_path,index=False)
display(threshold_table.head())
display(threshold_table.tail())

,threshold,precision,recall,f1,tn,fp,fn,tp,predicted_positive_rate,false_positive_rate,false_negative_rate,specificity,balanced_accuracy
0,0.01,0.684003,0.999769,0.812277,313,1999,1,4327,0.952711,0.864619,0.000231,0.135381,0.567575
1,0.02,0.693429,0.999769,0.818887,399,1913,1,4327,0.939759,0.827422,0.000231,0.172578,0.586173
2,0.03,0.698192,0.999538,0.822121,442,1870,2,4326,0.933133,0.808824,0.000462,0.191176,0.595357
3,0.04,0.705662,0.999307,0.827197,508,1804,3,4325,0.923042,0.780277,0.000693,0.219723,0.609515
4,0.05,0.713366,0.998845,0.832307,575,1737,5,4323,0.912651,0.751298,0.001155,0.248702,0.623774


,threshold,precision,recall,f1,tn,fp,fn,tp,predicted_positive_rate,false_positive_rate,false_negative_rate,specificity,balanced_accuracy
94,0.95,0.998871,0.817699,0.899250,2308,4,789,3539,0.533584,0.001730,0.182301,0.998270,0.907984
95,0.96,0.999432,0.813540,0.896956,2310,2,807,3521,0.530572,0.000865,0.186460,0.999135,0.906337
96,0.97,0.999708,0.789741,0.882406,2311,1,910,3418,0.514910,0.000433,0.210259,0.999567,0.894654
97,0.98,0.999693,0.753466,0.859289,2311,1,1067,3261,0.491265,0.000433,0.246534,0.999567,0.876517
98,0.99,1.000000,0.706331,0.827894,2312,0,1271,3057,0.460392,0.000000,0.293669,1.000000,0.853165


### What this cell does
Evaluates the difficult run over the same grid, selects the requested global decision candidates (including the minimum-FN extreme), and adds a run-aware development candidate that protects difficult-run recall.

### Why it matters
The globally most precise recall-constrained threshold can still fail the difficult run, so recommendation must consider both global recall and worst-run behavior.

### What to understand
The run-aware rule requires global recall at least 0.90 and difficult-run recall no lower than its current threshold-0.50 recall, then chooses the strongest F1 tradeoff.

In [4]:
difficult_run_id=prior_run_comparison.sort_values("official_pr_auc").iloc[0]["run_id"]
difficult_mask=validation_identifiers["run_id"].eq(difficult_run_id).to_numpy()
difficult_threshold_table=pd.DataFrame([threshold_metrics(y_validation.loc[difficult_mask],validation_probability[difficult_mask],t) for t in threshold_grid])
difficult_recall_at_050=float(difficult_threshold_table.loc[difficult_threshold_table["threshold"].eq(.50),"recall"].iloc[0])
eligible=threshold_table.loc[threshold_table["recall"].ge(.90)].copy(); assert len(eligible)>0
max_f1=threshold_table.sort_values(["f1","recall","precision"],ascending=[False,False,False]).iloc[0]
max_precision=eligible.sort_values(["precision","fp","f1"],ascending=[False,True,False]).iloc[0]
min_fp=eligible.sort_values(["fp","precision","f1"],ascending=[True,False,False]).iloc[0]
min_fn=eligible.sort_values(["fn","precision","f1"],ascending=[True,False,False]).iloc[0]
closest_050=eligible.assign(distance=lambda d:(d["threshold"]-.50).abs()).sort_values(["distance","f1"],ascending=[True,False]).iloc[0]
combined=threshold_table.merge(difficult_threshold_table[["threshold","recall"]].rename(columns={"recall":"difficult_recall"}),on="threshold")
run_aware_eligible=combined.loc[combined["recall"].ge(.90)&combined["difficult_recall"].ge(difficult_recall_at_050-1e-12)]
recommended=run_aware_eligible.sort_values(["f1","precision","threshold"],ascending=[False,False,False]).iloc[0]
candidate_rules=[("A_maximum_f1",max_f1),("B_max_precision_recall_ge_0_90",max_precision),
                 ("C_minimum_fp_recall_ge_0_90",min_fp),("D_closest_to_0_50_recall_ge_0_90",closest_050),
                 ("E_minimum_fn_recall_ge_0_90",min_fn),
                 ("recommended_run_aware",recommended)]
candidate_rows=[]
for rule,row in candidate_rules:
    t=float(row["threshold"]); global_metrics=threshold_table.loc[threshold_table["threshold"].eq(t)].iloc[0]; difficult_metrics=difficult_threshold_table.loc[difficult_threshold_table["threshold"].eq(t)].iloc[0]
    candidate_rows.append({"selection_rule":rule,**global_metrics.to_dict(),
                           **{f"difficult_run_{key}":difficult_metrics[key] for key in ["precision","recall","f1","fp","fn"]},
                           "difficult_run_id":difficult_run_id})
candidate_summary=pd.DataFrame(candidate_rows)
candidate_summary_path=REPORT_DIR/"threshold_candidate_summary.csv"
candidate_summary.to_csv(candidate_summary_path,index=False)
selected_rule_map={float(t):";".join(candidate_summary.loc[candidate_summary["threshold"].eq(t),"selection_rule"]) for t in candidate_summary["threshold"].unique()}
difficult_threshold_table["selected_candidate_rules"]=difficult_threshold_table["threshold"].map(selected_rule_map).fillna("")
difficult_path=REPORT_DIR/"threshold_difficult_run_regularized_lgbm.csv"
difficult_threshold_table.to_csv(difficult_path,index=False)
print(f"Difficult run: {difficult_run_id}; threshold-0.50 recall={difficult_recall_at_050:.4f}")
display(candidate_summary)

Difficult run: 1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d; threshold-0.50 recall=0.7330


,selection_rule,threshold,precision,recall,f1,tn,fp,fn,tp,predicted_positive_rate,false_positive_rate,false_negative_rate,specificity,balanced_accuracy,difficult_run_precision,difficult_run_recall,difficult_run_f1,difficult_run_fp,difficult_run_fn,difficult_run_id
0,A_maximum_f1,0.49,0.956389,0.942468,0.949377,2126.0,186.0,249.0,4079.0,0.642319,0.080450,0.057532,0.919550,0.931009,0.806701,0.746126,0.775232,150.0,213.0,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d
1,B_max_precision_recall_ge_0_90,0.63,0.982143,0.902264,0.940511,2241.0,71.0,423.0,3905.0,0.598795,0.030709,0.097736,0.969291,0.935777,0.906593,0.589988,0.714801,51.0,344.0,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d
2,C_minimum_fp_recall_ge_0_90,0.63,0.982143,0.902264,0.940511,2241.0,71.0,423.0,3905.0,0.598795,0.030709,0.097736,0.969291,0.935777,0.906593,0.589988,0.714801,51.0,344.0,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d
3,D_closest_to_0_50_recall_ge_0_90,0.50,0.958972,0.939695,0.949236,2138.0,174.0,261.0,4067.0,0.638705,0.075260,0.060305,0.924740,0.932218,0.816733,0.733015,0.772613,138.0,224.0,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d
4,E_minimum_fn_recall_ge_0_90,0.02,0.693429,0.999769,0.818887,399.0,1913.0,1.0,4327.0,0.939759,0.827422,0.000231,0.172578,0.586173,0.395568,1.000000,0.566892,1282.0,0.0,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d
5,recommended_run_aware,0.49,0.956389,0.942468,0.949377,2126.0,186.0,249.0,4079.0,0.642319,0.080450,0.057532,0.919550,0.931009,0.806701,0.746126,0.775232,150.0,213.0,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d


### What this cell does
Builds a compact business tradeoff table, checks each unique candidate threshold by validation machine, and saves row-aligned raw validation probabilities.

### Why it matters
Representative thresholds make alert-versus-miss costs understandable, while machine checks prevent a globally attractive threshold from hiding very low recall.

### What to understand
Machines with recall below 0.70 are flagged; with only three validation machines, these are diagnostic warnings rather than population-wide guarantees.

In [5]:
representative_thresholds=sorted(set([.20,.30,.40,.50,.60,.70,.80]+candidate_summary["threshold"].tolist()))
business_tradeoff=threshold_table.loc[threshold_table["threshold"].isin(representative_thresholds),["threshold","precision","recall","f1","fp","fn","predicted_positive_rate"]].copy()
business_tradeoff["candidate_rules"]=business_tradeoff["threshold"].map(selected_rule_map).fillna("representative_only")
business_path=REPORT_DIR/"threshold_business_tradeoff_regularized_lgbm.csv"
business_tradeoff.to_csv(business_path,index=False)
machine_rows=[]
for threshold in sorted(candidate_summary["threshold"].unique()):
    for machine_id,index in validation_identifiers.groupby("machine_id").groups.items():
        positions=np.asarray(list(index),dtype=int); result=threshold_metrics(y_validation.iloc[positions],validation_probability[positions],threshold)
        machine_rows.append({"candidate_rules":selected_rule_map[float(threshold)],"machine_id":machine_id,"rows":len(positions),
                             "positive_rate":float(y_validation.iloc[positions].mean()),**result,
                             "very_low_recall_warning":result["recall"]<.70})
threshold_by_machine=pd.DataFrame(machine_rows)
machine_path=REPORT_DIR/"threshold_by_machine_regularized_lgbm.csv"
threshold_by_machine.to_csv(machine_path,index=False)
validation_probabilities=validation_identifiers.reset_index(drop=True).copy()
validation_probabilities["true_target"]=y_validation.to_numpy(); validation_probabilities["regularized_lightgbm_probability"]=validation_probability
validation_probabilities["class_at_threshold_0_50"]=(validation_probability>=.50).astype(int)
probability_path=REPORT_DIR/"regularized_lgbm_validation_probabilities.csv"
validation_probabilities.to_csv(probability_path,index=False)
print("Business-style tradeoffs:"); display(business_tradeoff)
print("Candidate thresholds by machine:"); display(threshold_by_machine)

Business-style tradeoffs:


,threshold,precision,recall,f1,fp,fn,predicted_positive_rate,candidate_rules
1,0.02,0.693429,0.999769,0.818887,1913,1,0.939759,E_minimum_fn_recall_ge_0_90
19,0.20,0.824000,0.975739,0.893473,902,105,0.771837,representative_only
29,0.30,0.878947,0.964649,0.919806,575,153,0.715361,representative_only
39,0.40,0.928668,0.953558,0.940948,317,201,0.669277,representative_only
48,0.49,0.956389,0.942468,0.949377,186,249,0.642319,A_maximum_f1;recommended_run_aware
49,0.50,0.958972,0.939695,0.949236,174,261,0.638705,D_closest_to_0_50_recall_ge_0_90
59,0.60,0.976727,0.911506,0.942990,94,383,0.608283,representative_only
62,0.63,0.982143,0.902264,0.940511,71,423,0.598795,B_max_precision_recall_ge_0_90;C_minimum_fp_re...
69,0.70,0.990229,0.889787,0.937325,38,477,0.585693,representative_only
79,0.80,0.996046,0.873152,0.930559,15,549,0.571386,representative_only


Candidate thresholds by machine:


,candidate_rules,machine_id,rows,positive_rate,threshold,precision,recall,f1,tn,fp,fn,tp,predicted_positive_rate,false_positive_rate,false_negative_rate,specificity,balanced_accuracy,very_low_recall_warning
0,E_minimum_fn_recall_ge_0_90,7232bc533c21ce408d45d473,2190,0.383105,0.02,0.395568,1.000000,0.566892,69,1282,0,839,0.968493,0.948927,0.000000,0.051073,0.525537,False
1,E_minimum_fn_recall_ge_0_90,a0f8c86097e55fbfa506d057,3753,0.849987,0.02,0.849987,1.000000,0.918911,0,563,0,3190,1.000000,1.000000,0.000000,0.000000,0.500000,False
2,E_minimum_fn_recall_ge_0_90,d588df123ac0d0ce20b112ac,697,0.428981,0.02,0.814208,0.996656,0.896241,330,68,1,298,0.525108,0.170854,0.003344,0.829146,0.912901,False
3,A_maximum_f1;recommended_run_aware,7232bc533c21ce408d45d473,2190,0.383105,0.49,0.806701,0.746126,0.775232,1201,150,213,626,0.354338,0.111029,0.253874,0.888971,0.817549,False
4,A_maximum_f1;recommended_run_aware,a0f8c86097e55fbfa506d057,3753,0.849987,0.49,0.988725,0.989655,0.989190,527,36,33,3157,0.850786,0.063943,0.010345,0.936057,0.962856,False
5,A_maximum_f1;recommended_run_aware,d588df123ac0d0ce20b112ac,697,0.428981,0.49,1.000000,0.989967,0.994958,398,0,3,296,0.424677,0.000000,0.010033,1.000000,0.994983,False
6,D_closest_to_0_50_recall_ge_0_90,7232bc533c21ce408d45d473,2190,0.383105,0.50,0.816733,0.733015,0.772613,1213,138,224,615,0.343836,0.102147,0.266985,0.897853,0.815434,False
7,D_closest_to_0_50_recall_ge_0_90,a0f8c86097e55fbfa506d057,3753,0.849987,0.50,0.988722,0.989342,0.989032,527,36,34,3156,0.850520,0.063943,0.010658,0.936057,0.962699,False
8,D_closest_to_0_50_recall_ge_0_90,d588df123ac0d0ce20b112ac,697,0.428981,0.50,1.000000,0.989967,0.994958,398,0,3,296,0.424677,0.000000,0.010033,1.000000,0.994983,False
9,B_max_precision_recall_ge_0_90;C_minimum_fp_re...,7232bc533c21ce408d45d473,2190,0.383105,0.63,0.906593,0.589988,0.714801,1300,51,344,495,0.249315,0.037750,0.410012,0.962250,0.776119,True


### What this cell does
Creates six threshold figures and marks threshold 0.50, maximum-F1 threshold, and the run-aware recommended threshold.

### Why it matters
Visual curves reveal how precision, recall, false alerts, missed slowdowns, and difficult-run recall change together.

### What to understand
The marked recommendation is a development candidate only; no probability calibration or test-based confirmation is included.

In [6]:
max_f1_threshold=float(max_f1["threshold"]); recommended_threshold=float(recommended["threshold"])
markers=[(.50,"0.50","gray"),(max_f1_threshold,"Max F1","green"),(recommended_threshold,"Recommended","purple")]
figure_paths=[]
def threshold_line(column,title,ylabel,filename):
    fig,ax=plt.subplots(figsize=(8,5)); ax.plot(threshold_table["threshold"],threshold_table[column],linewidth=2)
    for x,label,color in markers: ax.axvline(x,color=color,linestyle="--",alpha=.8,label=f"{label}: {x:.2f}")
    ax.set(xlabel="Threshold",ylabel=ylabel,title=title,xlim=(.01,.99)); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); p=FIGURE_DIR/filename; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
threshold_line("precision","Precision versus threshold","Precision","precision_vs_threshold.png")
threshold_line("recall","Recall versus threshold","Recall","recall_vs_threshold.png")
threshold_line("f1","F1 versus threshold","F1","f1_vs_threshold.png")
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(threshold_table["threshold"],threshold_table["fp"],label="False positives"); ax.plot(threshold_table["threshold"],threshold_table["fn"],label="False negatives")
for x,label,color in markers: ax.axvline(x,color=color,linestyle="--",alpha=.8,label=f"{label}: {x:.2f}")
ax.set(xlabel="Threshold",ylabel="Rows",title="False positives and false negatives"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); p=FIGURE_DIR/"fp_fn_vs_threshold.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
fig,ax=plt.subplots(figsize=(7,6)); scatter=ax.scatter(threshold_table["recall"],threshold_table["precision"],c=threshold_table["threshold"],cmap="viridis"); fig.colorbar(scatter,ax=ax,label="Threshold")
for row,label,color in [(closest_050,"0.50","gray"),(max_f1,"Max F1","green"),(recommended,"Recommended","purple")]: ax.scatter(row["recall"],row["precision"],s=90,edgecolor=color,facecolor="none",linewidth=2,label=f"{label}: {row['threshold']:.2f}")
ax.set(xlabel="Recall",ylabel="Precision",title="Validation precision–recall threshold tradeoff"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); p=FIGURE_DIR/"precision_recall_tradeoff.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
fig,ax=plt.subplots(figsize=(8,5)); ax.plot(difficult_threshold_table["threshold"],difficult_threshold_table["recall"],linewidth=2)
for x,label,color in markers: ax.axvline(x,color=color,linestyle="--",alpha=.8,label=f"{label}: {x:.2f}")
ax.axhline(difficult_recall_at_050,color="red",linestyle=":",label=f"Recall at 0.50: {difficult_recall_at_050:.3f}"); ax.set(xlabel="Threshold",ylabel="Recall",title="Difficult-run recall versus threshold"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); p=FIGURE_DIR/"difficult_run_recall_vs_threshold.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
print(f"Saved {len(figure_paths)} figures under {FIGURE_DIR}")

Saved 6 figures under /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/threshold_optimization


### What this cell does
Reloads required outputs, verifies protected checksums and raw probabilities, and prints the requested candidate comparison and explicit stop condition.

### Why it matters
The final handoff must clearly separate a development threshold candidate from a production threshold and prove that calibration, retraining, and test evaluation did not occur.

### What to understand
Threshold 0.49 offers a small recall/F1 gain over 0.50 while protecting difficult-run recall; threshold 0.63 is globally precise but unacceptable for the difficult run.

In [7]:
required_outputs=[threshold_metrics_path,candidate_summary_path,machine_path,difficult_path,probability_path,business_path,*figure_paths]
assert all(path.exists() and path.stat().st_size>0 for path in required_outputs)
assert len(pd.read_csv(threshold_metrics_path))==99 and len(pd.read_csv(probability_path))==6_640
assert pd.read_csv(probability_path)["regularized_lightgbm_probability"].between(0,1).all()
hashes_after={name:sha256_file(path) for name,path in paths.items()}
protected_unchanged=hashes_before==hashes_after; assert protected_unchanged
at_050=threshold_table.loc[threshold_table["threshold"].eq(.50)].iloc[0]
recommended_difficult=difficult_threshold_table.loc[difficult_threshold_table["threshold"].eq(recommended_threshold)].iloc[0]
print("FINAL THRESHOLD-ANALYSIS REPORT")
print(f"Maximum-F1 threshold: {max_f1_threshold:.2f}; precision={max_f1.precision:.4f}, recall={max_f1.recall:.4f}, F1={max_f1.f1:.4f}, FP={int(max_f1.fp)}, FN={int(max_f1.fn)}")
print(f"Maximum-precision threshold with recall >= 0.90: {max_precision.threshold:.2f}; precision={max_precision.precision:.4f}, recall={max_precision.recall:.4f}, FP={int(max_precision.fp)}, FN={int(max_precision.fn)}")
print(f"Threshold 0.50: precision={at_050.precision:.4f}, recall={at_050.recall:.4f}, F1={at_050.f1:.4f}, FP={int(at_050.fp)}, FN={int(at_050.fn)}")
print(f"Run-aware development candidate: {recommended_threshold:.2f}; precision={recommended.precision:.4f}, recall={recommended.recall:.4f}, F1={recommended.f1:.4f}, FP={int(recommended.fp)}, FN={int(recommended.fn)}")
print(f"Difficult run at recommended threshold: precision={recommended_difficult.precision:.4f}, recall={recommended_difficult.recall:.4f}, F1={recommended_difficult.f1:.4f}, FP={int(recommended_difficult.fp)}, FN={int(recommended_difficult.fn)}")
print(f"The globally precise 0.63 threshold lowers difficult-run recall to {float(difficult_threshold_table.loc[difficult_threshold_table['threshold'].eq(.63),'recall'].iloc[0]):.4f}, so it is not recommended.")
print("Lower thresholds detect more slowdowns but generate more false alerts; higher thresholds reduce alerts but miss more slowdowns.")
print("Recommendation: use 0.49 only as the development candidate for later review; it is not a production threshold.")
print("WARNING: validation prevalence is unusually high (65.18%), so threshold behavior may not transfer to future operating data.")
print(f"Protected model, validation inputs, preprocessing, and prior reports unchanged: {protected_unchanged}")
print("STOP: no calibration, retraining, test evaluation, final model selection, SHAP, or dashboard work was performed.")

FINAL THRESHOLD-ANALYSIS REPORT
Maximum-F1 threshold: 0.49; precision=0.9564, recall=0.9425, F1=0.9494, FP=186, FN=249
Maximum-precision threshold with recall >= 0.90: 0.63; precision=0.9821, recall=0.9023, FP=71, FN=423
Threshold 0.50: precision=0.9590, recall=0.9397, F1=0.9492, FP=174, FN=261
Run-aware development candidate: 0.49; precision=0.9564, recall=0.9425, F1=0.9494, FP=186, FN=249
Difficult run at recommended threshold: precision=0.8067, recall=0.7461, F1=0.7752, FP=150, FN=213
The globally precise 0.63 threshold lowers difficult-run recall to 0.5900, so it is not recommended.
Lower thresholds detect more slowdowns but generate more false alerts; higher thresholds reduce alerts but miss more slowdowns.
Recommendation: use 0.49 only as the development candidate for later review; it is not a production threshold.
Protected model, validation inputs, preprocessing, and prior reports unchanged: True
STOP: no calibration, retraining, test evaluation, final model selection, SHAP, or